In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import calendar
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

BASE_DIR = Path("/home/sagemaker-user")
RAW_DIR = BASE_DIR / "Raw_Data"
CLEANED_DIR = BASE_DIR / "Data_Cleaned" / "Phase3_Final"

PORTFOLIOS = ["A", "B", "C", "D"]

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("CLEANED_DIR:", CLEANED_DIR)

print("A raw exists:", (RAW_DIR / "A_Interval.csv").exists())
print("A old cleaned exists:", (CLEANED_DIR / "A_Interval.csv").exists())

BASE_DIR: /home/sagemaker-user
RAW_DIR: /home/sagemaker-user/Raw_Data
CLEANED_DIR: /home/sagemaker-user/Data_Cleaned/Phase3_Final
A raw exists: True
A old cleaned exists: True


In [17]:
MONTH_MAP = {m.lower(): i for i, m in enumerate(calendar.month_name) if m}
MONTH_ABBR_MAP = {m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}

def parse_percent(series):
    s = series.astype(str).str.strip()
    s = s.str.replace("%", "", regex=False)
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return pd.to_numeric(s, errors="coerce") / 100.0

def parse_numeric(series):
    s = series.astype(str).str.strip()
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return pd.to_numeric(s, errors="coerce")

def parse_month_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in MONTH_MAP:
        return MONTH_MAP[s]
    if s in MONTH_ABBR_MAP:
        return MONTH_ABBR_MAP[s]
    try:
        v = int(float(s))
        return v if 1 <= v <= 12 else np.nan
    except:
        return np.nan

def interval_to_slot(interval_str):
    if pd.isna(interval_str):
        return np.nan
    s = str(interval_str).strip()
    try:
        parts = s.split(":")
        hour = int(parts[0])
        minute = int(parts[1])
        return hour * 2 + (1 if minute >= 30 else 0)
    except:
        return np.nan

def slot_to_interval(slot):
    hour = int(slot) // 2
    minute = 30 if int(slot) % 2 else 0
    return f"{hour:02d}:{minute:02d}:00"

def slot_to_hour(slot):
    return int(slot) // 2

def build_date_2025(month_num, day):
    try:
        return pd.Timestamp(year=2025, month=int(month_num), day=int(day))
    except:
        return pd.NaT

def most_complete_row(group, metric_cols):
    # keep the row with the highest number of non-null metric values
    scores = group[metric_cols].notna().sum(axis=1)
    idx = scores.idxmax()
    return group.loc[idx]

def add_time_features(df):
    df = df.copy()
    df["interval"] = df["slot_index"].apply(slot_to_interval)
    df["interval_start_hour"] = df["slot_index"].apply(slot_to_hour)
    df["is_weekend"] = df["date"].dt.dayofweek >= 5
    df["day_of_week"] = df["date"].dt.dayofweek
    df["month_num"] = df["date"].dt.month
    df["day"] = df["date"].dt.day

    def daypart(hour):
        if 0 <= hour < 6:
            return "overnight"
        elif 6 <= hour < 12:
            return "morning"
        elif 12 <= hour < 18:
            return "afternoon"
        else:
            return "evening"

    df["daypart"] = df["interval_start_hour"].apply(daypart)
    df["overnight_flag"] = (df["interval_start_hour"] < 6).astype(int)
    return df

In [26]:
def load_raw_interval(portfolio):
    file_path = RAW_DIR / f"{portfolio}_Interval.csv"
    df = pd.read_csv(file_path).copy()
    df.columns = [str(c).strip() for c in df.columns]

    expected = [
        "Month", "Day", "Interval", "Service Level",
        "Call Volume", "Abandoned Calls", "Abandoned Rate", "CCT"
    ]
    keep_cols = [c for c in expected if c in df.columns]
    df = df[keep_cols].copy()

    # Preserve originals for audit
    for col in keep_cols:
        df[f"{col}_raw"] = df[col]

    df["month_num"] = df["Month"].apply(parse_month_value)
    df["day"] = pd.to_numeric(df["Day"], errors="coerce")
    df["slot_index"] = df["Interval"].apply(interval_to_slot)
    df["date"] = [build_date_2025(m, d) for m, d in zip(df["month_num"], df["day"])]

    # Correct parsing by semantic type
    df["service_level"] = parse_percent(df["Service Level"]) if "Service Level" in df.columns else np.nan
    df["call_volume"] = parse_numeric(df["Call Volume"]) if "Call Volume" in df.columns else np.nan
    df["abandoned_calls"] = parse_numeric(df["Abandoned Calls"]) if "Abandoned Calls" in df.columns else np.nan
    df["abandoned_rate"] = parse_percent(df["Abandoned Rate"]) if "Abandoned Rate" in df.columns else np.nan
    df["cct"] = parse_numeric(df["CCT"]) if "CCT" in df.columns else np.nan

    # NEVER cast all numeric columns to int
    return df

In [53]:
def audit_raw_schema(df, portfolio):
    print(f"\n===== RAW SCHEMA AUDIT: {portfolio} =====")
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nDtypes:")
    print(df.dtypes)

    print("\nHead:")
    display(df.head(10))


def audit_parsed_schema(df, portfolio):
    print(f"\n===== PARSED SCHEMA AUDIT: {portfolio} =====")

    cols = [
        "month_num", "day", "slot_index", "date",
        "service_level", "call_volume", "abandoned_calls",
        "abandoned_rate", "cct"
    ]
    existing = [c for c in cols if c in df.columns]

    print("\nParsed dtypes:")
    print(df[existing].dtypes)

    print("\nPercent/rate range checks:")
    if "service_level" in df.columns:
        print("service_level min/max:", df["service_level"].min(), df["service_level"].max())
    if "abandoned_rate" in df.columns:
        print("abandoned_rate min/max:", df["abandoned_rate"].min(), df["abandoned_rate"].max())

    print("\nSample parsed rows:")
    preview_cols = [
        "Month", "Day", "Interval", "Service Level", "Call Volume",
        "Abandoned Calls", "Abandoned Rate", "CCT",
        "month_num", "day", "slot_index", "date",
        "service_level", "call_volume", "abandoned_calls",
        "abandoned_rate", "cct"
    ]
    preview_existing = [c for c in preview_cols if c in df.columns]
    display(df[preview_existing].head(12))

In [27]:
def audit_raw_interval(df, portfolio):
    print(f"\n===== RAW AUDIT: {portfolio} =====")

    print("\n--- BASIC SHAPE ---")
    print("Rows:", len(df))
    print("Unique dates:", df["date"].nunique())
    print("Date range:", df["date"].min(), "to", df["date"].max())

    print("\n--- DTYPE CHECK ---")
    cols_to_check = [
        "Month", "Day", "Interval", "Service Level", "Call Volume",
        "Abandoned Calls", "Abandoned Rate", "CCT",
        "month_num", "day", "slot_index", "date",
        "service_level", "call_volume", "abandoned_calls",
        "abandoned_rate", "cct"
    ]
    existing_cols = [c for c in cols_to_check if c in df.columns]
    print(df[existing_cols].dtypes)

    print("\n--- DATE / TIME PARSING CHECK ---")
    print("Missing month_num:", df["month_num"].isna().sum())
    print("Missing day:", df["day"].isna().sum())
    print("Missing slot_index:", df["slot_index"].isna().sum())
    print("Missing date:", df["date"].isna().sum())

    invalid_month = (~df["month_num"].between(1, 12)) & df["month_num"].notna()
    invalid_day = (~df["day"].between(1, 31)) & df["day"].notna()
    invalid_slot = (~df["slot_index"].between(0, 47)) & df["slot_index"].notna()

    print("Invalid month_num rows:", int(invalid_month.sum()))
    print("Invalid day rows:", int(invalid_day.sum()))
    print("Invalid slot_index rows:", int(invalid_slot.sum()))

    if df["date"].notna().any():
        print("\nDates by month:")
        print(df["date"].dt.month.value_counts(dropna=False).sort_index())

    bad_date_rows = df[df["date"].isna() | df["slot_index"].isna() | df["month_num"].isna() | df["day"].isna()]
    if len(bad_date_rows) > 0:
        print("\nSample rows with failed date/slot parsing:")
        display(
            bad_date_rows[
                [
                    "Month", "Day", "Interval",
                    "month_num", "day", "slot_index", "date"
                ]
            ].head(10)
        )

    print("\n--- DUPLICATE DATE-SLOT CHECK ---")
    dup_count = df.duplicated(subset=["date", "slot_index"], keep=False).sum()
    print("Duplicate date-slot rows:", dup_count)

    if dup_count > 0:
        print("\nSample duplicate date-slot rows:")
        display(
            df[df.duplicated(subset=["date", "slot_index"], keep=False)]
            .sort_values(["date", "slot_index"])
            .head(10)
        )

    print("\n--- SLOT COVERAGE CHECK ---")
    valid_date_slot = df.dropna(subset=["date", "slot_index"]).copy()
    if len(valid_date_slot) > 0:
        slots_per_day = valid_date_slot.groupby("date")["slot_index"].nunique().sort_values()
        print("Min slots/day:", slots_per_day.min())
        print("Max slots/day:", slots_per_day.max())
        print("Days with < 48 slots:", int((slots_per_day < 48).sum()))
        print("\nLowest slot-count days:")
        print(slots_per_day.head(10))
    else:
        print("No valid date-slot rows available for slot coverage check.")

    print("\n--- MISSING VALUES IN METRICS ---")
    metric_cols = ["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"]
    print(df[metric_cols].isna().sum())

    print("\n--- PERCENT COLUMN RANGE CHECK ---")
    if "service_level" in df.columns:
        print("service_level min/max:", df["service_level"].min(), df["service_level"].max())
        bad_sl = ((df["service_level"] < 0) | (df["service_level"] > 1)) & df["service_level"].notna()
        print("service_level outside [0,1]:", int(bad_sl.sum()))

    if "abandoned_rate" in df.columns:
        print("abandoned_rate min/max:", df["abandoned_rate"].min(), df["abandoned_rate"].max())
        bad_ar = ((df["abandoned_rate"] < 0) | (df["abandoned_rate"] > 1)) & df["abandoned_rate"].notna()
        print("abandoned_rate outside [0,1]:", int(bad_ar.sum()))

    print("\n--- NUMERIC SANITY CHECK ---")
    for col in ["call_volume", "abandoned_calls", "cct"]:
        if col in df.columns:
            print(f"{col} min/max:", df[col].min(), df[col].max())

    suspicious_numeric = df[
        ((df["call_volume"] < 0) & df["call_volume"].notna()) |
        ((df["abandoned_calls"] < 0) & df["abandoned_calls"].notna()) |
        ((df["cct"] < 0) & df["cct"].notna())
    ]
    print("Rows with negative numeric values:", len(suspicious_numeric))
    if len(suspicious_numeric) > 0:
        display(
            suspicious_numeric[
                [
                    "Month", "Day", "Interval",
                    "call_volume", "abandoned_calls", "abandoned_rate", "cct"
                ]
            ].head(10)
        )

In [37]:
def deduplicate_interval(df):
    metric_cols = ["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"]

    valid = df.dropna(subset=["date", "slot_index"]).copy()
    valid["slot_index"] = valid["slot_index"].astype(int)

    # score rows by completeness
    valid["non_null_score"] = valid[metric_cols].notna().sum(axis=1)

    # identify duplicate keys first
    key_counts = (
        valid.groupby(["date", "slot_index"])
             .size()
             .reset_index(name="row_count")
    )

    dup_keys = key_counts[key_counts["row_count"] > 1].copy()

    dup_rows = valid.merge(
        dup_keys[["date", "slot_index"]],
        on=["date", "slot_index"],
        how="inner"
    ).copy()

    # keep the most complete row per date-slot
    deduped = (
        valid.sort_values(
            ["date", "slot_index", "non_null_score"],
            ascending=[True, True, False]
        )
        .drop_duplicates(subset=["date", "slot_index"], keep="first")
        .copy()
    )

    # mark which surviving rows came from duplicate groups
    deduped = deduped.merge(
        dup_keys.assign(was_duplicate_resolved=1)[["date", "slot_index", "was_duplicate_resolved"]],
        on=["date", "slot_index"],
        how="left"
    )

    deduped["was_duplicate_resolved"] = deduped["was_duplicate_resolved"].fillna(0).astype(int)
    deduped = deduped.drop(columns=["non_null_score"], errors="ignore")

    return deduped.sort_values(["date", "slot_index"]).reset_index(drop=True), dup_rows

In [38]:
print("duplicate raw rows resolved:", len(dup_rows))

duplicate raw rows resolved: 0


In [39]:
print("unique duplicate date-slot keys:", dup_rows[["date", "slot_index"]].drop_duplicates().shape[0])

unique duplicate date-slot keys: 0


In [40]:
def build_full_grid(deduped_df):
    all_dates = pd.date_range("2025-04-01", "2025-06-30", freq="D")
    full_grid = pd.MultiIndex.from_product(
        [all_dates, range(48)],
        names=["date", "slot_index"]
    ).to_frame(index=False)

    merged = full_grid.merge(
        deduped_df,
        on=["date", "slot_index"],
        how="left",
        suffixes=("", "_raw")
    )

    merged["slot_missing_flag"] = merged["call_volume"].isna() & merged["abandoned_calls"].isna() & merged["abandoned_rate"].isna() & merged["cct"].isna()
    merged["was_duplicate_resolved"] = merged["was_duplicate_resolved"].fillna(0).astype(int)

    return merged

In [41]:
def audit_post_grid(df, portfolio):
    print(f"\n===== POST-GRID AUDIT: {portfolio} =====")
    print("Rows:", len(df))
    print("Expected rows:", 91 * 48)
    print("Unique dates:", df["date"].nunique())
    print("Slot_missing_flag count:", int(df["slot_missing_flag"].sum()))

    slots_per_day = df.groupby("date")["slot_index"].nunique()
    print("Min slots/day:", slots_per_day.min())
    print("Max slots/day:", slots_per_day.max())

    print("\nMissing metrics before imputation:")
    print(df[["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"]].isna().sum())

In [58]:
def safe_impute_interval(df):
    df = df.copy()
    df = add_time_features(df)

    # preserve originals
    for col in ["call_volume", "abandoned_calls", "abandoned_rate", "cct", "service_level"]:
        df[f"orig_{col}"] = df[col]

    # flags
    for col in ["call_volume", "abandoned_calls", "abandoned_rate", "cct", "service_level"]:
        df[f"was_imputed_{col}"] = 0

    df["was_recomputed_abandoned_rate"] = 0
    df["was_fixed_zero_volume_row"] = 0
    df["was_fixed_bad_abandoned_count"] = 0

    # conservative grouped fill helper
    def grouped_fill(series_name):
        s = df[series_name]

        g1 = df.groupby(["day_of_week", "slot_index"])[series_name].median()
        g2 = df.groupby(["day_of_week", "daypart"])[series_name].median()
        g3 = df.groupby(["slot_index"])[series_name].median()
        g4 = df.groupby(["daypart"])[series_name].median()
        g5 = df[series_name].median()

        def fill_row(row):
            if pd.notna(row[series_name]):
                return row[series_name]

            v = g1.get((row["day_of_week"], row["slot_index"]), np.nan)
            if pd.isna(v):
                v = g2.get((row["day_of_week"], row["daypart"]), np.nan)
            if pd.isna(v):
                v = g3.get(row["slot_index"], np.nan)
            if pd.isna(v):
                v = g4.get(row["daypart"], np.nan)
            if pd.isna(v):
                v = g5
            return v

        miss = s.isna()
        df.loc[miss, series_name] = df.loc[miss].apply(fill_row, axis=1)
        df.loc[miss, f"was_imputed_{series_name}"] = 1

    # -----------------------------
    # 1. CALL VOLUME
    # -----------------------------
    grouped_fill("call_volume")
    df["call_volume"] = df["call_volume"].clip(lower=0).round()

    # -----------------------------
    # 2. SERVICE LEVEL
    # -----------------------------
    grouped_fill("service_level")
    df["service_level"] = df["service_level"].clip(0, 1)

   # -----------------------------
    # 3. CCT
    # -----------------------------
    grouped_fill("cct")
    df["cct"] = df["cct"].clip(lower=0)

    # optional local fallback for any remaining CCT gaps
    df["cct"] = df.groupby("date")["cct"].transform(lambda s: s.ffill().bfill())
    df["cct"] = df["cct"].fillna(df["cct"].median())

    # only cap clearly absurd overnight CCT on tiny volume
    extreme_mask = (
        (df["interval_start_hour"] < 6) &
        (df["call_volume"] <= 2) &
        (df["cct"] > 2400)
    )

    ref_cct = df.groupby(["day_of_week", "slot_index"])["cct"].median()

    def replace_extreme_cct(row):
        v = ref_cct.get((row["day_of_week"], row["slot_index"]), np.nan)
        return row["cct"] if pd.isna(v) else v

    df.loc[extreme_mask, "cct"] = df.loc[extreme_mask].apply(replace_extreme_cct, axis=1)

    # -----------------------------
    # 4. ABANDON RATE
    # -----------------------------
    # fill grouped, but preserve valid observed values
    grouped_fill("abandoned_rate")
    df["abandoned_rate"] = df["abandoned_rate"].clip(0, 1)

    # -----------------------------
    # 5. EDGE CASE: ZERO / NULL VOLUME
    # -----------------------------
    zero_vol = df["call_volume"].isna() | (df["call_volume"] <= 0)

    # if volume is zero, counts and rate must both be zero
    df.loc[zero_vol, "abandoned_calls"] = 0
    df.loc[zero_vol, "abandoned_rate"] = 0.0
    df.loc[zero_vol, "was_fixed_zero_volume_row"] = 1

    # -----------------------------
    # 6. ABANDONED CALLS
    # -----------------------------
    # only for rows where volume > 0
    positive_vol = df["call_volume"] > 0

    # if abandoned_calls missing and rate available, derive from volume * rate
    miss_ac = df["abandoned_calls"].isna() & positive_vol
    df.loc[miss_ac, "abandoned_calls"] = (
        df.loc[miss_ac, "call_volume"] * df.loc[miss_ac, "abandoned_rate"]
    ).round()
    df.loc[miss_ac, "was_imputed_abandoned_calls"] = 1

    # if abandoned_calls still missing somehow, use grouped fill and clamp
    still_miss_ac = df["abandoned_calls"].isna() & positive_vol
    if still_miss_ac.any():
        g1 = df.groupby(["day_of_week", "slot_index"])["abandoned_calls"].median()
        g2 = df.groupby(["day_of_week", "daypart"])["abandoned_calls"].median()
        g3 = df.groupby("slot_index")["abandoned_calls"].median()
        g4 = df.groupby("daypart")["abandoned_calls"].median()
        g5 = df["abandoned_calls"].median()

        def fill_ac_row(row):
            v = g1.get((row["day_of_week"], row["slot_index"]), np.nan)
            if pd.isna(v):
                v = g2.get((row["day_of_week"], row["daypart"]), np.nan)
            if pd.isna(v):
                v = g3.get(row["slot_index"], np.nan)
            if pd.isna(v):
                v = g4.get(row["daypart"], np.nan)
            if pd.isna(v):
                v = g5
            return v

        df.loc[still_miss_ac, "abandoned_calls"] = df.loc[still_miss_ac].apply(fill_ac_row, axis=1).round()
        df.loc[still_miss_ac, "was_imputed_abandoned_calls"] = 1

    df["abandoned_calls"] = df["abandoned_calls"].fillna(0).clip(lower=0).round()

    # impossible rows: abandoned_calls > call_volume
    bad_counts = (df["abandoned_calls"] > df["call_volume"]) & positive_vol
    df.loc[bad_counts, "abandoned_calls"] = df.loc[bad_counts, "call_volume"]
    df.loc[bad_counts, "was_fixed_bad_abandoned_count"] = 1

    # -----------------------------
    # 7. RECOMPUTE RATE ONLY WHEN NECESSARY
    # -----------------------------
    need_rate = (
        df["abandoned_rate"].isna() |
        (df["abandoned_rate"] < 0) |
        (df["abandoned_rate"] > 1) |
        bad_counts
    ) & positive_vol

    df.loc[need_rate, "abandoned_rate"] = (
        df.loc[need_rate, "abandoned_calls"] /
        df.loc[need_rate, "call_volume"]
    ).clip(0, 1)

    df.loc[need_rate, "was_recomputed_abandoned_rate"] = 1

    # final consistency cleanup
    df.loc[~positive_vol, "abandoned_calls"] = 0
    df.loc[~positive_vol, "abandoned_rate"] = 0.0

    return df

In [59]:
def audit_after_imputation(df, portfolio):
    print(f"\n===== FINAL CLEAN AUDIT: {portfolio} =====")
    print("Rows:", len(df))
    print("Unique dates:", df["date"].nunique() if "date" in df.columns else "date column missing")

    print("\nMissing values after cleaning:")
    cols = ["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct", "interval", "month_num"]
    existing_cols = [c for c in cols if c in df.columns]
    missing_cols = [c for c in cols if c not in df.columns]

    if existing_cols:
        print(df[existing_cols].isna().sum())
    if missing_cols:
        print("Columns not present in this dataframe:", missing_cols)

    print("\nImputation counts:")
    imp_cols = {
        "call_volume": "was_imputed_call_volume",
        "abandoned_calls": "was_imputed_abandoned_calls",
        "abandoned_rate": "was_imputed_abandoned_rate",
        "cct": "was_imputed_cct",
        "service_level": "was_imputed_service_level",
    }
    out = {}
    for k, v in imp_cols.items():
        out[k] = int(df[v].sum()) if v in df.columns else "flag missing"
    print(out)

    print("\nSanity checks:")
    print("Any duplicate date-slot rows:", df.duplicated(subset=["date", "slot_index"]).any() if {"date", "slot_index"}.issubset(df.columns) else "cannot check")
    print("Any negative call_volume:", (df["call_volume"] < 0).any() if "call_volume" in df.columns else "cannot check")
    print("Any negative abandoned_calls:", (df["abandoned_calls"] < 0).any() if "abandoned_calls" in df.columns else "cannot check")
    print("Any negative cct:", (df["cct"] < 0).any() if "cct" in df.columns else "cannot check")
    print("Any abandoned_rate outside [0,1]:", (((df["abandoned_rate"] < 0) | (df["abandoned_rate"] > 1)).any()) if "abandoned_rate" in df.columns else "cannot check")
    print("Any abandoned_calls > call_volume:", (df["abandoned_calls"] > df["call_volume"]).any() if {"abandoned_calls", "call_volume"}.issubset(df.columns) else "cannot check")

In [60]:
def format_for_export(df):
    out = df.copy()

    # keep clean export columns
    out = out[
        [
            "month_num", "day", "interval", "service_level",
            "call_volume", "abandoned_calls", "abandoned_rate", "cct",
            "date", "slot_index", "interval_start_hour", "is_weekend",
            "day_of_week", "daypart", "overnight_flag",
            "slot_missing_flag", "was_duplicate_resolved",
            "was_imputed_call_volume", "was_imputed_abandoned_calls",
            "was_imputed_abandoned_rate", "was_imputed_cct",
            "was_imputed_service_level"
        ]
    ].copy()

    return out

In [61]:
needed_functions = [
    "load_raw_interval",
    "audit_raw_interval",
    "deduplicate_interval",
    "build_full_grid",
    "audit_post_grid",
    "safe_impute_interval",
    "audit_after_imputation",
    "format_for_export"
]

for fn in needed_functions:
    print(fn, "->", fn in globals())

load_raw_interval -> True
audit_raw_interval -> True
deduplicate_interval -> True
build_full_grid -> True
audit_post_grid -> True
safe_impute_interval -> True
audit_after_imputation -> True
format_for_export -> True


In [62]:
portfolio = "A"

raw_df = load_raw_interval(portfolio)

# raw-stage audits only
audit_raw_schema(raw_df, portfolio)
audit_parsed_schema(raw_df, portfolio)
audit_raw_interval(raw_df, portfolio)

# dedupe + grid
deduped_df, dup_rows = deduplicate_interval(raw_df)
print("Duplicate raw rows resolved:", len(dup_rows))
print("Unique duplicate date-slot keys:", dup_rows[["date", "slot_index"]].drop_duplicates().shape[0])

grid_df = build_full_grid(deduped_df)
audit_post_grid(grid_df, portfolio)

# clean + final audit
clean_df = safe_impute_interval(grid_df)
audit_after_imputation(clean_df, portfolio)

# targeted repair audit
print("\n===== TARGETED REPAIR COUNTS =====")
print("was_fixed_zero_volume_row:", int(clean_df["was_fixed_zero_volume_row"].sum()))
print("was_fixed_bad_abandoned_count:", int(clean_df["was_fixed_bad_abandoned_count"].sum()))
print("was_recomputed_abandoned_rate:", int(clean_df["was_recomputed_abandoned_rate"].sum()))

# optional quick sanity snapshots
print("\nRows with zero-volume fixes:")
display(
    clean_df.loc[clean_df["was_fixed_zero_volume_row"] == 1, [
        "date", "slot_index", "interval", "call_volume",
        "abandoned_calls", "abandoned_rate", "cct"
    ]].head(10)
)

print("\nRows with bad abandoned-count fixes:")
display(
    clean_df.loc[clean_df["was_fixed_bad_abandoned_count"] == 1, [
        "date", "slot_index", "interval", "call_volume",
        "abandoned_calls", "abandoned_rate", "cct"
    ]].head(10)
)

print("\nRows where abandoned_rate was recomputed:")
display(
    clean_df.loc[clean_df["was_recomputed_abandoned_rate"] == 1, [
        "date", "slot_index", "interval", "call_volume",
        "abandoned_calls", "abandoned_rate", "cct"
    ]].head(10)
)

export_df = format_for_export(clean_df)
display(export_df.head(20))


===== RAW SCHEMA AUDIT: A =====

Columns:
['Month', 'Day', 'Interval', 'Service Level', 'Call Volume', 'Abandoned Calls', 'Abandoned Rate', 'CCT', 'Month_raw', 'Day_raw', 'Interval_raw', 'Service Level_raw', 'Call Volume_raw', 'Abandoned Calls_raw', 'Abandoned Rate_raw', 'CCT_raw', 'month_num', 'day', 'slot_index', 'date', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']

Dtypes:
Month                          object
Day                           float64
Interval                       object
Service Level                  object
Call Volume                   float64
Abandoned Calls               float64
Abandoned Rate                 object
CCT                            object
Month_raw                      object
Day_raw                       float64
Interval_raw                   object
Service Level_raw              object
Call Volume_raw               float64
Abandoned Calls_raw           float64
Abandoned Rate_raw             object
CCT_raw            

,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT,Month_raw,Day_raw,Interval_raw,Service Level_raw,Call Volume_raw,Abandoned Calls_raw,Abandoned Rate_raw,CCT_raw,month_num,day,slot_index,date,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6,4.0,1.0,0.0,2025-04-01,1.0,5.0,0.0,0.0,137.60
1,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4,4.0,1.0,1.0,2025-04-01,1.0,5.0,0.0,0.0,263.40
2,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25,4.0,1.0,2.0,2025-04-01,1.0,4.0,0.0,0.0,333.25
3,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170,4.0,1.0,3.0,2025-04-01,1.0,3.0,0.0,0.0,170.00
4,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667,4.0,1.0,4.0,2025-04-01,1.0,1.0,0.0,0.0,667.00
5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5,4.0,1.0,5.0,2025-04-01,1.0,4.0,0.0,0.0,416.50
6,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128,4.0,1.0,6.0,2025-04-01,1.0,1.0,0.0,0.0,128.00
7,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158,4.0,1.0,7.0,2025-04-01,1.0,4.0,0.0,0.0,158.00
8,April,1.0,5:00,NaN,NaN,NaN,0.00%,59,April,1.0,5:00,NaN,NaN,NaN,0.00%,59,4.0,1.0,10.0,2025-04-01,NaN,NaN,NaN,0.0,59.00
9,April,1.0,5:30,NaN,NaN,NaN,0.00%,178,April,1.0,5:30,NaN,NaN,NaN,0.00%,178,4.0,1.0,11.0,2025-04-01,NaN,NaN,NaN,0.0,178.00



===== PARSED SCHEMA AUDIT: A =====

Parsed dtypes:
month_num                 float64
day                       float64
slot_index                float64
date               datetime64[ns]
service_level             float64
call_volume               float64
abandoned_calls           float64
abandoned_rate            float64
cct                       float64
dtype: object

Percent/rate range checks:
service_level min/max: 0.0 1.0
abandoned_rate min/max: 0.0 1.0

Sample parsed rows:


,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT,month_num,day,slot_index,date,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6,4.0,1.0,0.0,2025-04-01,1.0,5.0,0.0,0.0,137.60
1,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4,4.0,1.0,1.0,2025-04-01,1.0,5.0,0.0,0.0,263.40
2,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25,4.0,1.0,2.0,2025-04-01,1.0,4.0,0.0,0.0,333.25
3,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170,4.0,1.0,3.0,2025-04-01,1.0,3.0,0.0,0.0,170.00
4,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667,4.0,1.0,4.0,2025-04-01,1.0,1.0,0.0,0.0,667.00
5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5,4.0,1.0,5.0,2025-04-01,1.0,4.0,0.0,0.0,416.50
6,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128,4.0,1.0,6.0,2025-04-01,1.0,1.0,0.0,0.0,128.00
7,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158,4.0,1.0,7.0,2025-04-01,1.0,4.0,0.0,0.0,158.00
8,April,1.0,5:00,NaN,NaN,NaN,0.00%,59,4.0,1.0,10.0,2025-04-01,NaN,NaN,NaN,0.0,59.00
9,April,1.0,5:30,NaN,NaN,NaN,0.00%,178,4.0,1.0,11.0,2025-04-01,NaN,NaN,NaN,0.0,178.00



===== RAW AUDIT: A =====

--- BASIC SHAPE ---
Rows: 4084
Unique dates: 91
Date range: 2025-04-01 00:00:00 to 2025-06-30 00:00:00

--- DTYPE CHECK ---
Month                      object
Day                       float64
Interval                   object
Service Level              object
Call Volume               float64
Abandoned Calls           float64
Abandoned Rate             object
CCT                        object
month_num                 float64
day                       float64
slot_index                float64
date               datetime64[ns]
service_level             float64
call_volume               float64
abandoned_calls           float64
abandoned_rate            float64
cct                       float64
dtype: object

--- DATE / TIME PARSING CHECK ---
Missing month_num: 1
Missing day: 1
Missing slot_index: 8
Missing date: 1
Invalid month_num rows: 0
Invalid day rows: 0
Invalid slot_index rows: 0

Dates by month:
date
4.0    1336
5.0    1394
6.0    1353
NaN       1
Name:

...


Rows with bad abandoned-count fixes:


,date,slot_index,interval,call_volume,abandoned_calls,abandoned_rate,cct



Rows where abandoned_rate was recomputed:


,date,slot_index,interval,call_volume,abandoned_calls,abandoned_rate,cct


,month_num,day,interval,service_level,call_volume,abandoned_calls,abandoned_rate,cct,date,slot_index,interval_start_hour,is_weekend,day_of_week,daypart,overnight_flag,slot_missing_flag,was_duplicate_resolved,was_imputed_call_volume,was_imputed_abandoned_calls,was_imputed_abandoned_rate,was_imputed_cct,was_imputed_service_level
0,4,1,00:00:00,1.00000,5.0,0.0,0.0000,137.60,2025-04-01,0,0,False,1,overnight,1,False,0,0,0,0,0,0
1,4,1,00:30:00,1.00000,5.0,0.0,0.0000,263.40,2025-04-01,1,0,False,1,overnight,1,False,0,0,0,0,0,0
2,4,1,01:00:00,1.00000,4.0,0.0,0.0000,333.25,2025-04-01,2,1,False,1,overnight,1,False,0,0,0,0,0,0
3,4,1,01:30:00,1.00000,3.0,0.0,0.0000,170.00,2025-04-01,3,1,False,1,overnight,1,False,0,0,0,0,0,0
4,4,1,02:00:00,1.00000,1.0,0.0,0.0000,667.00,2025-04-01,4,2,False,1,overnight,1,False,0,0,0,0,0,0
5,4,1,02:30:00,1.00000,4.0,0.0,0.0000,416.50,2025-04-01,5,2,False,1,overnight,1,False,0,0,0,0,0,0
6,4,1,03:00:00,1.00000,1.0,0.0,0.0000,128.00,2025-04-01,6,3,False,1,overnight,1,False,0,0,0,0,0,0
7,4,1,03:30:00,1.00000,4.0,0.0,0.0000,158.00,2025-04-01,7,3,False,1,overnight,1,False,0,0,0,0,0,0
8,4,1,04:00:00,1.00000,1.0,0.0,0.0000,229.00,2025-04-01,8,4,False,1,overnight,1,True,0,1,1,1,1,1
9,4,1,04:30:00,1.00000,2.0,0.0,0.0000,298.67,2025-04-01,9,4,False,1,overnight,1,True,0,1,1,1,1,1


In [63]:
print(raw_df[["Service Level_raw", "service_level", "Abandoned Rate_raw", "abandoned_rate"]].head(12))
print("service_level min/max:", raw_df["service_level"].min(), raw_df["service_level"].max())
print("abandoned_rate min/max:", raw_df["abandoned_rate"].min(), raw_df["abandoned_rate"].max())

   Service Level_raw  service_level Abandoned Rate_raw  abandoned_rate
0            100.00%            1.0              0.00%             0.0
1            100.00%            1.0              0.00%             0.0
2            100.00%            1.0              0.00%             0.0
3            100.00%            1.0              0.00%             0.0
4            100.00%            1.0              0.00%             0.0
5            100.00%            1.0              0.00%             0.0
6            100.00%            1.0              0.00%             0.0
7            100.00%            1.0              0.00%             0.0
8                NaN            NaN              0.00%             0.0
9                NaN            NaN              0.00%             0.0
10               NaN            NaN              0.00%             0.0
11               NaN            NaN             50.00%             0.5
service_level min/max: 0.0 1.0
abandoned_rate min/max: 0.0 1.0


In [64]:
out_path = CLEANED_DIR / "A_Interval_V2.csv"
export_df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /home/sagemaker-user/Data_Cleaned/Phase3_Final/A_Interval_V2.csv


In [65]:
old_df = pd.read_csv(CLEANED_DIR / "A_Interval.csv")
new_df = pd.read_csv(CLEANED_DIR / "A_Interval_V2.csv")

print("OLD shape:", old_df.shape)
print("NEW shape:", new_df.shape)

print("\nOLD missing:")
print(old_df.isna().sum().sort_values(ascending=False).head(15))

print("\nNEW missing:")
print(new_df.isna().sum().sort_values(ascending=False).head(15))

OLD shape: (4368, 21)
NEW shape: (4368, 22)

OLD missing:
service_level          4368
month_num              1820
interval               1820
date                      0
slot_index                0
day                       0
month                     0
abandoned_calls           0
call_volume               0
abandoned_rate            0
cct                       0
interval_start_hour       0
overnight_flag            0
is_weekend                0
slot_missing_flag         0
dtype: int64

NEW missing:
month_num              0
day                    0
interval               0
service_level          0
call_volume            0
abandoned_calls        0
abandoned_rate         0
cct                    0
date                   0
slot_index             0
interval_start_hour    0
is_weekend             0
day_of_week            0
daypart                0
overnight_flag         0
dtype: int64


In [66]:
results_summary = []

for portfolio in ["A", "B", "C", "D"]:
    print("\n" + "=" * 100)
    print(f"PROCESSING PORTFOLIO {portfolio}")
    print("=" * 100)

    # -----------------------------
    # 1. LOAD + RAW AUDITS
    # -----------------------------
    raw_df = load_raw_interval(portfolio)

    audit_raw_schema(raw_df, portfolio)
    audit_parsed_schema(raw_df, portfolio)
    audit_raw_interval(raw_df, portfolio)

    # -----------------------------
    # 2. DEDUPE + GRID
    # -----------------------------
    deduped_df, dup_rows = deduplicate_interval(raw_df)
    print("\nDuplicate raw rows resolved:", len(dup_rows))
    print("Unique duplicate date-slot keys:", dup_rows[["date", "slot_index"]].drop_duplicates().shape[0])

    grid_df = build_full_grid(deduped_df)
    audit_post_grid(grid_df, portfolio)

    # -----------------------------
    # 3. CLEAN + FINAL AUDIT
    # -----------------------------
    clean_df = safe_impute_interval(grid_df)
    audit_after_imputation(clean_df, portfolio)

    # -----------------------------
    # 4. TARGETED REPAIR COUNTS
    # -----------------------------
    print("\n===== TARGETED REPAIR COUNTS =====")
    zero_fix = int(clean_df["was_fixed_zero_volume_row"].sum()) if "was_fixed_zero_volume_row" in clean_df.columns else 0
    bad_count_fix = int(clean_df["was_fixed_bad_abandoned_count"].sum()) if "was_fixed_bad_abandoned_count" in clean_df.columns else 0
    rate_recompute = int(clean_df["was_recomputed_abandoned_rate"].sum()) if "was_recomputed_abandoned_rate" in clean_df.columns else 0

    print("was_fixed_zero_volume_row:", zero_fix)
    print("was_fixed_bad_abandoned_count:", bad_count_fix)
    print("was_recomputed_abandoned_rate:", rate_recompute)

    print("\nSample rows with zero-volume fixes:")
    display(
        clean_df.loc[clean_df["was_fixed_zero_volume_row"] == 1, [
            "date", "slot_index", "interval", "call_volume",
            "abandoned_calls", "abandoned_rate", "cct"
        ]].head(10)
    )

    print("\nSample rows with bad abandoned-count fixes:")
    display(
        clean_df.loc[clean_df["was_fixed_bad_abandoned_count"] == 1, [
            "date", "slot_index", "interval", "call_volume",
            "abandoned_calls", "abandoned_rate", "cct"
        ]].head(10)
    )

    print("\nSample rows where abandoned_rate was recomputed:")
    display(
        clean_df.loc[clean_df["was_recomputed_abandoned_rate"] == 1, [
            "date", "slot_index", "interval", "call_volume",
            "abandoned_calls", "abandoned_rate", "cct"
        ]].head(10)
    )

    # -----------------------------
    # 5. EXPORT NEW CLEANED FILE
    # -----------------------------
    export_df = format_for_export(clean_df)

    new_path = CLEANED_DIR / f"{portfolio}_Interval_V2.csv"
    export_df.to_csv(new_path, index=False)
    print("\nSaved new cleaned file:", new_path)

    # -----------------------------
    # 6. COMPARE OLD VS NEW CLEANED
    # -----------------------------
    old_path = CLEANED_DIR / f"{portfolio}_Interval.csv"

    if old_path.exists():
        old_df = pd.read_csv(old_path)
        new_df = pd.read_csv(new_path)

        print("\n===== OLD VS NEW CLEANED COMPARISON =====")
        print("OLD shape:", old_df.shape)
        print("NEW shape:", new_df.shape)

        old_missing = old_df.isna().sum()
        new_missing = new_df.isna().sum()

        print("\nTop OLD missing columns:")
        print(old_missing.sort_values(ascending=False).head(15))

        print("\nTop NEW missing columns:")
        print(new_missing.sort_values(ascending=False).head(15))

        results_summary.append({
            "portfolio": portfolio,
            "old_rows": old_df.shape[0],
            "new_rows": new_df.shape[0],
            "old_total_missing": int(old_missing.sum()),
            "new_total_missing": int(new_missing.sum()),
            "zero_volume_fixes": zero_fix,
            "bad_abandoned_count_fixes": bad_count_fix,
            "recomputed_abandoned_rate": rate_recompute
        })
    else:
        print(f"\nOld cleaned file not found for {portfolio}: {old_path}")

        results_summary.append({
            "portfolio": portfolio,
            "old_rows": None,
            "new_rows": export_df.shape[0],
            "old_total_missing": None,
            "new_total_missing": int(export_df.isna().sum().sum()),
            "zero_volume_fixes": zero_fix,
            "bad_abandoned_count_fixes": bad_count_fix,
            "recomputed_abandoned_rate": rate_recompute
        })


PROCESSING PORTFOLIO A

===== RAW SCHEMA AUDIT: A =====

Columns:
['Month', 'Day', 'Interval', 'Service Level', 'Call Volume', 'Abandoned Calls', 'Abandoned Rate', 'CCT', 'Month_raw', 'Day_raw', 'Interval_raw', 'Service Level_raw', 'Call Volume_raw', 'Abandoned Calls_raw', 'Abandoned Rate_raw', 'CCT_raw', 'month_num', 'day', 'slot_index', 'date', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']

Dtypes:
Month                          object
Day                           float64
Interval                       object
Service Level                  object
Call Volume                   float64
Abandoned Calls               float64
Abandoned Rate                 object
CCT                            object
Month_raw                      object
Day_raw                       float64
Interval_raw                   object
Service Level_raw              object
Call Volume_raw               float64
Abandoned Calls_raw           float64
Abandoned Rate_raw             ob

,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT,Month_raw,Day_raw,Interval_raw,Service Level_raw,Call Volume_raw,Abandoned Calls_raw,Abandoned Rate_raw,CCT_raw,month_num,day,slot_index,date,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6,4.0,1.0,0.0,2025-04-01,1.0,5.0,0.0,0.0,137.60
1,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4,4.0,1.0,1.0,2025-04-01,1.0,5.0,0.0,0.0,263.40
2,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25,4.0,1.0,2.0,2025-04-01,1.0,4.0,0.0,0.0,333.25
3,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170,4.0,1.0,3.0,2025-04-01,1.0,3.0,0.0,0.0,170.00
4,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667,4.0,1.0,4.0,2025-04-01,1.0,1.0,0.0,0.0,667.00
5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5,4.0,1.0,5.0,2025-04-01,1.0,4.0,0.0,0.0,416.50
6,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128,4.0,1.0,6.0,2025-04-01,1.0,1.0,0.0,0.0,128.00
7,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158,4.0,1.0,7.0,2025-04-01,1.0,4.0,0.0,0.0,158.00
8,April,1.0,5:00,NaN,NaN,NaN,0.00%,59,April,1.0,5:00,NaN,NaN,NaN,0.00%,59,4.0,1.0,10.0,2025-04-01,NaN,NaN,NaN,0.0,59.00
9,April,1.0,5:30,NaN,NaN,NaN,0.00%,178,April,1.0,5:30,NaN,NaN,NaN,0.00%,178,4.0,1.0,11.0,2025-04-01,NaN,NaN,NaN,0.0,178.00



===== PARSED SCHEMA AUDIT: A =====

Parsed dtypes:
month_num                 float64
day                       float64
slot_index                float64
date               datetime64[ns]
service_level             float64
call_volume               float64
abandoned_calls           float64
abandoned_rate            float64
cct                       float64
dtype: object

Percent/rate range checks:
service_level min/max: 0.0 1.0
abandoned_rate min/max: 0.0 1.0

Sample parsed rows:


,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT,month_num,day,slot_index,date,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6,4.0,1.0,0.0,2025-04-01,1.0,5.0,0.0,0.0,137.60
1,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4,4.0,1.0,1.0,2025-04-01,1.0,5.0,0.0,0.0,263.40
2,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25,4.0,1.0,2.0,2025-04-01,1.0,4.0,0.0,0.0,333.25
3,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170,4.0,1.0,3.0,2025-04-01,1.0,3.0,0.0,0.0,170.00
4,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667,4.0,1.0,4.0,2025-04-01,1.0,1.0,0.0,0.0,667.00
5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5,4.0,1.0,5.0,2025-04-01,1.0,4.0,0.0,0.0,416.50
6,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128,4.0,1.0,6.0,2025-04-01,1.0,1.0,0.0,0.0,128.00
7,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158,4.0,1.0,7.0,2025-04-01,1.0,4.0,0.0,0.0,158.00
8,April,1.0,5:00,NaN,NaN,NaN,0.00%,59,4.0,1.0,10.0,2025-04-01,NaN,NaN,NaN,0.0,59.00
9,April,1.0,5:30,NaN,NaN,NaN,0.00%,178,4.0,1.0,11.0,2025-04-01,NaN,NaN,NaN,0.0,178.00



===== RAW AUDIT: A =====

--- BASIC SHAPE ---
Rows: 4084
Unique dates: 91
Date range: 2025-04-01 00:00:00 to 2025-06-30 00:00:00

--- DTYPE CHECK ---
Month                      object
Day                       float64
Interval                   object
Service Level              object
Call Volume               float64
Abandoned Calls           float64
Abandoned Rate             object
CCT                        object
month_num                 float64
day                       float64
slot_index                float64
date               datetime64[ns]
service_level             float64
call_volume               float64
abandoned_calls           float64
abandoned_rate            float64
cct                       float64
dtype: object

--- DATE / TIME PARSING CHECK ---
Missing month_num: 1
Missing day: 1
Missing slot_index: 8
Missing date: 1
Invalid month_num rows: 0
Invalid day rows: 0
Invalid slot_index rows: 0

Dates by month:
date
4.0    1336
5.0    1394
6.0    1353
NaN       1
Name:

...


Sample rows with bad abandoned-count fixes:


,date,slot_index,interval,call_volume,abandoned_calls,abandoned_rate,cct



Sample rows where abandoned_rate was recomputed:


,date,slot_index,interval,call_volume,abandoned_calls,abandoned_rate,cct



Saved new cleaned file: /home/sagemaker-user/Data_Cleaned/Phase3_Final/D_Interval_V2.csv

===== OLD VS NEW CLEANED COMPARISON =====
OLD shape: (4368, 21)
NEW shape: (4368, 22)

Top OLD missing columns:
service_level          4368
month_num              1820
interval               1820
date                      0
slot_index                0
day                       0
month                     0
abandoned_calls           0
call_volume               0
abandoned_rate            0
cct                       0
interval_start_hour       0
overnight_flag            0
is_weekend                0
slot_missing_flag         0
dtype: int64

Top NEW missing columns:
month_num              0
day                    0
interval               0
service_level          0
call_volume            0
abandoned_calls        0
abandoned_rate         0
cct                    0
date                   0
slot_index             0
interval_start_hour    0
is_weekend             0
day_of_week            0
daypart      

In [67]:
summary_df = pd.DataFrame(results_summary)
print("\n===== FINAL SUMMARY ACROSS ALL PORTFOLIOS =====")
display(summary_df)


===== FINAL SUMMARY ACROSS ALL PORTFOLIOS =====


,portfolio,old_rows,new_rows,old_total_missing,new_total_missing,zero_volume_fixes,bad_abandoned_count_fixes,recomputed_abandoned_rate
0,A,4368,4368,8008,0,52,0,0
1,B,4368,4368,8018,0,23,0,0
2,C,4368,4368,8016,0,1,0,0
3,D,4368,4368,8008,0,254,0,0
